#**Logistic Regression**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/AI Training/Datasets/Copy of neo.csv')

In [ ]:
df.head()

,id,name,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,sentry_object,absolute_magnitude,hazardous
0,2162635,162635 (2000 SS164),1.198271,2.679415,13569.249224,5.483974e+07,Earth,False,16.73,False
1,2277475,277475 (2005 WK4),0.265800,0.594347,73588.726663,6.143813e+07,Earth,False,20.00,True
2,2512244,512244 (2015 YE18),0.722030,1.614507,114258.692129,4.979872e+07,Earth,False,17.83,False
3,3596030,(2012 BV13),0.096506,0.215794,24764.303138,2.543497e+07,Earth,False,22.20,False
4,3667127,(2014 GE35),0.255009,0.570217,42737.733765,4.627557e+07,Earth,False,20.09,True


In [ ]:
df.isnull().sum()

,0
id,0
name,0
est_diameter_min,0
est_diameter_max,0
relative_velocity,0
miss_distance,0
orbiting_body,0
sentry_object,0
absolute_magnitude,0
hazardous,0


In [ ]:
df.columns.to_list()

['id',
 'name',
 'est_diameter_min',
 'est_diameter_max',
 'relative_velocity',
 'miss_distance',
 'orbiting_body',
 'sentry_object',
 'absolute_magnitude',
 'hazardous']

In [ ]:
x=df[['est_diameter_min','est_diameter_max','relative_velocity','miss_distance','absolute_magnitude']]
y=df['hazardous']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
x_train.head()

,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,absolute_magnitude
2639,0.024241,0.054205,43303.999094,4.814117e+07,25.20
29138,0.030238,0.067615,21770.790211,5.646643e+07,24.72
36927,0.201630,0.450858,109358.123029,6.435051e+07,20.60
61855,0.160160,0.358129,78494.609756,5.595780e+07,21.10
15916,0.006991,0.015633,19077.749486,3.834648e+07,27.90


In [ ]:
def remove_outlier(x_train, x_test):
  !pip install feature-engine
  from feature_engine.outliers import Winsorizer
  winsor = Winsorizer(
    capping_method="quantiles",
    tail="both",
    fold=0.01
)

  for col in x_train.columns.to_list():

    x_train[col] = winsor.fit_transform(x_train[[col]])
    x_test[col] = winsor.transform(x_test[[col]])

  return (x_train, x_test)

In [ ]:
x_train, x_test=remove_outlier(x_train, x_test)

In [ ]:
def scale_data(x_train, x_test):
  from sklearn.preprocessing import StandardScaler
  ss=StandardScaler()
  for i in x_train.columns.to_list():
    x_train[i]=ss.fit_transform(x_train[[i]])
    x_test[i]=ss.transform(x_test[[i]])

  return(x_train, x_test)

In [ ]:
x_train, x_test=scale_data(x_train, x_test)

In [ ]:
def decompose(x_train, x_test):
  from sklearn.decomposition import PCA
  pca=PCA(n_components=2)
  x_train=pca.fit_transform(x_train)
  x_test=pca.transform(x_test)

  return (x_train, x_test)

In [ ]:
x_train, x_test=decompose(x_train, x_test)

In [ ]:
def clear_imbalance(x_train, y_train):
  from imblearn.over_sampling import RandomOverSampler
  os=RandomOverSampler()

  x_train_sampled, y_train_sampled = os.fit_resample(x_train, y_train)

  return (x_train_sampled, y_train_sampled)

In [ ]:
x_train, y_train=clear_imbalance(x_train, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(class_weight='balanced')

model.fit(x_train, y_train)

y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

print('accuracy_score')
a=accuracy_score(y_test, y_pred)
print(a)
'printconfusion_matrix'
print(confusion_matrix(y_test, y_pred))
print()
print('classification_report')
print(classification_report(y_test, y_pred))

accuracy_score
0.8008586525759577
[[13160  3240]
 [  378  1390]]

classification_report
              precision    recall  f1-score   support

       False       0.97      0.80      0.88     16400
        True       0.30      0.79      0.43      1768

    accuracy                           0.80     18168
   macro avg       0.64      0.79      0.66     18168
weighted avg       0.91      0.80      0.84     18168

